In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('청년정책목록_전체.csv', encoding='utf-8')

In [3]:
df.head(3)

,plcyNo,bscPlanCycl,bscPlanPlcyWayNo,bscPlanFcsAsmtNo,bscPlanAsmtNo,pvsnInstGroupCd,plcyPvsnMthdCd,plcyAprvSttsCd,plcyNm,plcyKywdNm,...,rgtrHghrkInstCd,rgtrHghrkInstCdNm,zipCd,plcyMajorCd,jobCd,schoolCd,aplyYmd,frstRegDt,lastMdfcnDt,sbizCd
0,20250521005400110863,1,3,10,23,54001,42013.0,44002,주거안정장학금,"보조금,교육지원",...,1492000,고용노동부,"11110,11140,11170,11200,11215,11230,11260,1129...",0011009,0013010,0049005,20250523 ~ 20250623,2025-05-21 15:36:19,2025-05-21 17:13:42,0014010
1,20250520005400210862,1,4,14,35,54002,42010.0,44002,부산 청년돌봄이음,바우처,...,6260000,부산광역시,"26110,26140,26170,26200,26230,26260,26290,2632...",0011009,0013010,0049010,NaN,2025-05-20 13:35:27,2025-05-21 14:49:00,0014010
2,20250519005400210855,1,4,15,39,54002,42005.0,44002,서울 청년 마음건강 지원사업,맞춤형상담서비스,...,6110000,서울특별시,"11110,11140,11170,11200,11215,11230,11260,1129...",0011009,0013010,0049010,20250201 ~ 20250930\N20250414 ~ 20250417,2025-05-19 20:39:07,2025-05-21 14:24:31,0014009


In [4]:
col = ['정책번호', '기본계획차수', '기본계획정책방향번호', '기본계획중점과제번호', '기본계획과제번호', '제공기관그룹코드', '정책제공방법코드', '정책승인상태코드', '정책명', '정책키워드명',
       '정책설명내용', '정책대분류명', '정책중분류명', '정책지원내용', '주관기관코드', '주관기관코드명', '주관기관담당자명', '운영기관코드', '운영기관코드명', '운영기관담당자명',
       '지원규모제한여부', '신청기간구분코드', '사업기간구분코드', '사업기간시작일자', '사업기간종료일자', '사업기간기타내용', '정책신청방법내용', '심사방법내용', '신청URL주소', '제출서류내용',
       '기타사항내용', '참고URL주소1', '참고URL주소2', '지원규모수', '지원도착순서여부', '지원대상최소연령', '지원대상최대연령', '지원대상연령제한여부', '결혼상태코드', '소득조건구분코드',
       '소득최소금액', '소득최대금액', '소득기타내용', '추가신청자격조건내용', '참여제안대상내용', '조회수', '등록자기관코드', '등록자기관코드명', '등록자상위기관코드', '등록자상위기관코드명',
       '등록자최상위기관코드', '등록자최상위기관코드명', '정책거주지역코드', '정책전공요건코드', '정책취업요건코드', '정책학력요건코드', '신청기간', '최초등록일시', '최종수정일시', '정책특화요건코드',
       ]   

In [5]:
df.columns = col

In [6]:
df_code = pd.read_csv('청년정책_전체코드매핑.csv', encoding='utf-8')

In [7]:
# df_code의 코드그룹명과 df 컬럼이 일치하는 경우 코드를 코드명으로 매핑

# df_code에서 존재하는 코드그룹명 목록 확인
unique_code_groups = df_code['코드그룹명'].unique()
print(f"코드 그룹 종류: {len(unique_code_groups)}개")
print(unique_code_groups)

# df 컬럼 중에서 코드그룹명과 일치하는 컬럼 확인
matching_columns = [col for col in df.columns if col in unique_code_groups]
print(f"\n데이터프레임에서 매핑 가능한 컬럼: {len(matching_columns)}개")
print(matching_columns)


코드 그룹 종류: 11개
['전공조건코드' '취업상태코드' '특수분야코드' '정책제공방법코드' '소득조건구분코드' '정책승인상태코드' '자격학력코드'
 '제공기관그룹코드' '결혼상태코드' '사업기간구분코드' '신청기간구분코드']

데이터프레임에서 매핑 가능한 컬럼: 7개
['제공기관그룹코드', '정책제공방법코드', '정책승인상태코드', '신청기간구분코드', '사업기간구분코드', '결혼상태코드', '소득조건구분코드']


In [8]:
def map_codes_to_names(df, df_code, column_name):
    # 해당 코드그룹에 대한 코드-코드명 매핑 딕셔너리 생성
    code_map = df_code[df_code['코드그룹명'] == column_name].set_index('코드')['코드명'].to_dict()
    
    # 원래 컬럼값을 저장 (로깅 목적)
    original_values = df[column_name].copy()
    
    # 코드를 코드명으로 변환 (기존 컬럼 값 대체)
    df[column_name] = df[column_name].map(code_map)
    
    # 매핑된 결과 요약
    mapped_count = df[column_name].notna().sum()
    print(f"{column_name}: 총 {len(df)}개 중 {mapped_count}개 매핑 완료 ({mapped_count/len(df)*100:.2f}%)")
    
    # 매핑된 샘플 몇 개 보여주기
    if mapped_count > 0:
        sample_df = pd.DataFrame({
            '원래 코드': original_values.head(5),
            '매핑된 이름': df[column_name].head(5)
        })
        print(sample_df)
    
    return df

In [9]:
# 매칭되는 모든 컬럼에 대해 매핑 실행
for column in matching_columns:
    df = map_codes_to_names(df, df_code, column)
    print('-' * 50)

print("\n매핑 완료!")
# 일부 매핑 결과 확인
print("매핑된 데이터 샘플:")
if matching_columns:
    print(df[matching_columns].head())
else:
    print("매핑할 컬럼이 없습니다.")

제공기관그룹코드: 총 3458개 중 3458개 매핑 완료 (100.00%)
   원래 코드 매핑된 이름
0  54001   중앙부처
1  54002    지자체
2  54002    지자체
3  54002    지자체
4  54002    지자체
--------------------------------------------------
정책제공방법코드: 총 3458개 중 623개 매핑 완료 (18.02%)
     원래 코드    매핑된 이름
0  42013.0        기타
1  42010.0       바우처
2  42005.0  계약(위탁운영)
3  42005.0  계약(위탁운영)
4  42002.0      프로그램
--------------------------------------------------
정책승인상태코드: 총 3458개 중 3458개 매핑 완료 (100.00%)
   원래 코드 매핑된 이름
0  44002     승인
1  44002     승인
2  44002     승인
3  44002     승인
4  44002     승인
--------------------------------------------------
신청기간구분코드: 총 3458개 중 3458개 매핑 완료 (100.00%)
   원래 코드 매핑된 이름
0  57001   특정기간
1  57002     상시
2  57001   특정기간
3  57001   특정기간
4  57001   특정기간
--------------------------------------------------
사업기간구분코드: 총 3458개 중 619개 매핑 완료 (17.90%)
     원래 코드 매핑된 이름
0  56002.0     기타
1  56001.0   특정기간
2  56001.0   특정기간
3  56001.0   특정기간
4  56001.0   특정기간
--------------------------------------------------
결혼상태코드: 총 3458개 중

In [10]:
# 1. 전공조건코드 그룹에서 코드-코드명 매핑 딕셔너리 생성
major_code_map = df_code[df_code['코드그룹명'] == '전공조건코드'].set_index('코드')['코드명'].to_dict()

# 2. 코드 변환 함수 정의 - 쉼표로 구분된 코드들도 처리
def transform_major_code(code_value):
    # None이나 NaN인 경우 그대로 반환
    if pd.isna(code_value):
        return code_value
        
    # 쉼표로 구분된 여러 코드가 있는지 확인
    if isinstance(code_value, str) and ',' in code_value:
        # 쉼표로 구분된 각 코드를 처리
        codes = code_value.split(',')
        transformed_codes = []
        
        for code in codes:
            if code.strip().startswith('00'):
                try:
                    # '00' 제거 후 정수로 변환
                    transformed_code = int(code.strip()[2:])
                    # 매핑 딕셔너리에서 코드명 찾기
                    code_name = major_code_map.get(transformed_code)
                    if code_name:
                        transformed_codes.append(code_name)
                except ValueError:
                    # 변환 실패 시 원래 코드 유지
                    transformed_codes.append(code.strip())
            else:
                transformed_codes.append(code.strip())
                
        # 변환된 코드명들을 쉼표로 연결하여 반환
        return ', '.join(transformed_codes) if transformed_codes else code_value
    
    # 단일 코드인 경우    
    elif isinstance(code_value, str) and code_value.startswith('00'):
        try:
            # '00' 제거 후 정수로 변환
            transformed_code = int(code_value[2:])
            # 매핑 딕셔너리에서 코드명 찾기
            return major_code_map.get(transformed_code, code_value)
        except ValueError:
            # 변환 실패 시 원래 코드 유지
            return code_value
    
    # 그 외의 경우 원래 값 그대로 반환
    return code_value

# 3. 원래 값 저장 (비교용)
original_major_codes = df['정책전공요건코드'].copy()

# 4. 변환 함수 적용하여 코드명으로 매핑
df['정책전공요건코드'] = df['정책전공요건코드'].apply(transform_major_code)

# 5. 매핑 결과 확인
mapped_count = df['정책전공요건코드'].notna().sum()
print(f"정책전공요건코드: 총 {len(df)}개 중 {mapped_count}개 매핑 완료 ({mapped_count/len(df)*100:.2f}%)")

# 6. 매핑 전후 비교 샘플 출력
sample_df = pd.DataFrame({
    '원래 코드': original_major_codes.head(10),
    '매핑된 이름': df['정책전공요건코드'].head(10)
})
print("\n매핑 전후 샘플 비교:")
print(sample_df)

# 7. 매핑된 값들의 분포 확인
df['정책전공요건코드'].value_counts()

정책전공요건코드: 총 3458개 중 3458개 매핑 완료 (100.00%)

매핑 전후 샘플 비교:
     원래 코드 매핑된 이름
0  0011009   제한없음
1  0011009   제한없음
2  0011009   제한없음
3  0011009   제한없음
4  0011009   제한없음
5  0011009   제한없음
6  0011009   제한없음
7  0011009   제한없음
8  0011009   제한없음
9  0011009   제한없음


정책전공요건코드
제한없음                                              3322
인문계열, 사회계열, 상경계열, 이학계열, 공학계열, 예체능계열, 농산업계열          70
기타                                                  13
이학계열, 공학계열                                          12
예체능계열                                                9
인문계열, 사회계열, 상경계열, 이학계열, 공학계열, 예체능계열, 농산업계열, 기타       8
농산업계열                                                8
공학계열                                                 7
인문계열, 사회계열                                           3
상경계열, 이학계열, 공학계열                                     2
인문계열, 사회계열, 상경계열, 이학계열, 공학계열, 예체능계열                  1
공학계열, 예체능계열, 기타                                      1
이학계열, 공학계열, 농산업계열                                    1
이학계열                                                 1
Name: count, dtype: int64

In [11]:
# 1. 자격학력코드 그룹에서 코드-코드명 매핑 딕셔너리 생성
education_code_map = df_code[df_code['코드그룹명'] == '자격학력코드'].set_index('코드')['코드명'].to_dict()

# 2. 코드 변환 함수 정의 - 쉼표로 구분된 코드들도 처리
def transform_education_code(code_value):
    # None이나 NaN인 경우 그대로 반환
    if pd.isna(code_value):
        return code_value
        
    # 쉼표로 구분된 여러 코드가 있는지 확인
    if isinstance(code_value, str) and ',' in code_value:
        # 쉼표로 구분된 각 코드를 처리
        codes = code_value.split(',')
        transformed_codes = []
        
        for code in codes:
            if code.strip().startswith('00'):
                try:
                    # '00' 제거 후 정수로 변환
                    transformed_code = int(code.strip()[2:])
                    # 매핑 딕셔너리에서 코드명 찾기
                    code_name = education_code_map.get(transformed_code)
                    if code_name:
                        transformed_codes.append(code_name)
                except ValueError:
                    # 변환 실패 시 원래 코드 유지
                    transformed_codes.append(code.strip())
            else:
                transformed_codes.append(code.strip())
                
        # 변환된 코드명들을 쉼표로 연결하여 반환
        return ', '.join(transformed_codes) if transformed_codes else code_value
    
    # 단일 코드인 경우    
    elif isinstance(code_value, str) and code_value.startswith('00'):
        try:
            # '00' 제거 후 정수로 변환
            transformed_code = int(code_value[2:])
            # 매핑 딕셔너리에서 코드명 찾기
            return education_code_map.get(transformed_code, code_value)
        except ValueError:
            # 변환 실패 시 원래 코드 유지
            return code_value
    
    # 그 외의 경우 원래 값 그대로 반환
    return code_value

# 3. 원래 값 저장 (비교용)
original_education_codes = df['정책학력요건코드'].copy()

# 4. 변환 함수 적용하여 코드명으로 매핑
df['정책학력요건코드'] = df['정책학력요건코드'].apply(transform_education_code)

# 5. 매핑 결과 확인
mapped_count = df['정책학력요건코드'].notna().sum()
print(f"정책학력요건코드: 총 {len(df)}개 중 {mapped_count}개 매핑 완료 ({mapped_count/len(df)*100:.2f}%)")

# 6. 매핑 전후 비교 샘플 출력
sample_df = pd.DataFrame({
    '원래 코드': original_education_codes.head(10),
    '매핑된 이름': df['정책학력요건코드'].head(10)
})
print("\n매핑 전후 샘플 비교:")
print(sample_df)

# 7. 매핑된 값들의 분포 확인
df['정책학력요건코드'].value_counts()

정책학력요건코드: 총 3458개 중 3458개 매핑 완료 (100.00%)

매핑 전후 샘플 비교:
     원래 코드 매핑된 이름
0  0049005  대학 재학
1  0049010   제한없음
2  0049010   제한없음
3  0049010   제한없음
4  0049010   제한없음
5  0049010   제한없음
6  0049010   제한없음
7  0049010   제한없음
8  0049010   제한없음
9  0049010   제한없음


정책학력요건코드
제한없음                                                     3014
대학 재학                                                      94
고졸 미만, 고교 재학, 고졸 예정, 고교 졸업, 대학 재학, 대졸 예정, 대학 졸업, 석·박사      67
대학 재학, 대졸 예정                                               54
대학 재학, 대졸 예정, 대학 졸업, 석·박사                                  19
                                                         ... 
고졸 예정, 고교 졸업, 대학 재학, 대졸 예정                                  1
고졸 미만, 고교 재학                                                1
고졸 미만, 고졸 예정, 대졸 예정, 석·박사                                   1
고교 재학, 고교 졸업, 대학 재학, 대졸 예정                                  1
고졸 예정, 고교 졸업                                                1
Name: count, Length: 66, dtype: int64

In [12]:
# 1. 취업상태코드 그룹에서 코드-코드명 매핑 딕셔너리 생성
employment_code_map = df_code[df_code['코드그룹명'] == '취업상태코드'].set_index('코드')['코드명'].to_dict()

# 2. 코드 변환 함수 정의 - 쉼표로 구분된 코드들도 처리
def transform_employment_code(code_value):
    # None이나 NaN인 경우 그대로 반환
    if pd.isna(code_value):
        return code_value
        
    # 쉼표로 구분된 여러 코드가 있는지 확인
    if isinstance(code_value, str) and ',' in code_value:
        # 쉼표로 구분된 각 코드를 처리
        codes = code_value.split(',')
        transformed_codes = []
        
        for code in codes:
            if code.strip().startswith('00'):
                try:
                    # '00' 제거 후 정수로 변환
                    transformed_code = int(code.strip()[2:])
                    # 매핑 딕셔너리에서 코드명 찾기
                    code_name = employment_code_map.get(transformed_code)
                    if code_name:
                        transformed_codes.append(code_name)
                except ValueError:
                    # 변환 실패 시 원래 코드 유지
                    transformed_codes.append(code.strip())
            else:
                transformed_codes.append(code.strip())
                
        # 변환된 코드명들을 쉼표로 연결하여 반환
        return ', '.join(transformed_codes) if transformed_codes else code_value
    
    # 단일 코드인 경우    
    elif isinstance(code_value, str) and code_value.startswith('00'):
        try:
            # '00' 제거 후 정수로 변환
            transformed_code = int(code_value[2:])
            # 매핑 딕셔너리에서 코드명 찾기
            return employment_code_map.get(transformed_code, code_value)
        except ValueError:
            # 변환 실패 시 원래 코드 유지
            return code_value
    
    # 그 외의 경우 원래 값 그대로 반환
    return code_value

# 3. 원래 값 저장 (비교용)
original_employment_codes = df['정책취업요건코드'].copy()

# 4. 변환 함수 적용하여 코드명으로 매핑
df['정책취업요건코드'] = df['정책취업요건코드'].apply(transform_employment_code)

# 5. 매핑 결과 확인
mapped_count = df['정책취업요건코드'].notna().sum()
print(f"정책취업요건코드: 총 {len(df)}개 중 {mapped_count}개 매핑 완료 ({mapped_count/len(df)*100:.2f}%)")

# 6. 매핑 전후 비교 샘플 출력
sample_df = pd.DataFrame({
    '원래 코드': original_employment_codes.head(10),
    '매핑된 이름': df['정책취업요건코드'].head(10)
})
print("\n매핑 전후 샘플 비교:")
print(sample_df)

# 7. 매핑된 값들의 분포 확인
df['정책취업요건코드'].value_counts()

정책취업요건코드: 총 3458개 중 3458개 매핑 완료 (100.00%)

매핑 전후 샘플 비교:
             원래 코드     매핑된 이름
0          0013010       제한없음
1          0013010       제한없음
2          0013010       제한없음
3          0013001        재직자
4          0013010       제한없음
5          0013010       제한없음
6          0013010       제한없음
7          0013010       제한없음
8          0013010       제한없음
9  0013001,0013003  재직자, 미취업자


정책취업요건코드
제한없음                                        2392
미취업자                                         447
(예비)창업자                                      196
재직자                                           86
자영업자, (예비)창업자                                 69
                                            ... 
자영업자, 미취업자, 일용근로자                              1
재직자, 일용근로자, 단기근로자, 영농종사자                       1
재직자, 미취업자, 일용근로자, 단기근로자                        1
재직자, 영농종사자                                     1
미취업자, 프리랜서, 일용근로자, (예비)창업자, 단기근로자, 영농종사자       1
Name: count, Length: 69, dtype: int64

In [13]:
# 1. 특수분야코드 그룹에서 코드-코드명 매핑 딕셔너리 생성
special_code_map = df_code[df_code['코드그룹명'] == '특수분야코드'].set_index('코드')['코드명'].to_dict()

# 2. 코드 변환 함수 정의 - 쉼표로 구분된 코드들도 처리
def transform_special_code(code_value):
    # None이나 NaN인 경우 그대로 반환
    if pd.isna(code_value):
        return code_value
        
    # 쉼표로 구분된 여러 코드가 있는지 확인
    if isinstance(code_value, str) and ',' in code_value:
        # 쉼표로 구분된 각 코드를 처리
        codes = code_value.split(',')
        transformed_codes = []
        
        for code in codes:
            if code.strip().startswith('00'):
                try:
                    # '00' 제거 후 정수로 변환
                    transformed_code = int(code.strip()[2:])
                    # 매핑 딕셔너리에서 코드명 찾기
                    code_name = special_code_map.get(transformed_code)
                    if code_name:
                        transformed_codes.append(code_name)
                except ValueError:
                    # 변환 실패 시 원래 코드 유지
                    transformed_codes.append(code.strip())
            else:
                transformed_codes.append(code.strip())
                
        # 변환된 코드명들을 쉼표로 연결하여 반환
        return ', '.join(transformed_codes) if transformed_codes else code_value
    
    # 단일 코드인 경우    
    elif isinstance(code_value, str) and code_value.startswith('00'):
        try:
            # '00' 제거 후 정수로 변환
            transformed_code = int(code_value[2:])
            # 매핑 딕셔너리에서 코드명 찾기
            return special_code_map.get(transformed_code, code_value)
        except ValueError:
            # 변환 실패 시 원래 코드 유지
            return code_value
    
    # 그 외의 경우 원래 값 그대로 반환
    return code_value

# 3. 원래 값 저장 (비교용)
original_special_codes = df['정책특화요건코드'].copy()

# 4. 변환 함수 적용하여 코드명으로 매핑
df['정책특화요건코드'] = df['정책특화요건코드'].apply(transform_special_code)

# 5. 매핑 결과 확인
mapped_count = df['정책특화요건코드'].notna().sum()
print(f"정책특화요건코드: 총 {len(df)}개 중 {mapped_count}개 매핑 완료 ({mapped_count/len(df)*100:.2f}%)")

# 6. 매핑 전후 비교 샘플 출력
sample_df = pd.DataFrame({
    '원래 코드': original_special_codes.head(10),
    '매핑된 이름': df['정책특화요건코드'].head(10)
})
print("\n매핑 전후 샘플 비교:")
print(sample_df)

# 7. 매핑된 값들의 분포 확인
df['정책특화요건코드'].value_counts()

정책특화요건코드: 총 3458개 중 3458개 매핑 완료 (100.00%)

매핑 전후 샘플 비교:
     원래 코드 매핑된 이름
0  0014010   제한없음
1  0014010   제한없음
2  0014009     기타
3  0014009     기타
4  0014010   제한없음
5  0014010   제한없음
6  0014010   제한없음
7  0014010   제한없음
8  0014010   제한없음
9  0014001   중소기업


정책특화요건코드
제한없음                                     3111
중소기업                                       66
농업인                                        45
기초생활수급자                                    41
중소기업, 여성, 기초생활수급자, 장애인, 농업인, 군인, 지역인재      39
지역인재                                       25
기타                                         24
여성                                         22
군인                                         19
장애인                                         9
여성, 기초생활수급자                                 8
기초생활수급자, 장애인                                8
기초생활수급자, 장애인, 지역인재                          5
기초생활수급자, 지역인재                               5
중소기업, 지역인재                                  3
여성, 기초생활수급자, 장애인                            3
농업인, 지역인재                                   3
한부모가정, 기타                                   2
중소기업, 기초생활수급자, 농업인, 지역인재                    2
여성, 기초생활수급자, 장애인, 농업인, 지역인재                 2
중소기업, 여성                                    2
중소기업, 기초생활수급자, 지역인재      

In [14]:
df.iloc[0]

정책번호                                        20250521005400110863
기본계획차수                                                         1
기본계획정책방향번호                                                     3
기본계획중점과제번호                                                    10
기본계획과제번호                                                      23
제공기관그룹코드                                                    중앙부처
정책제공방법코드                                                      기타
정책승인상태코드                                                      승인
정책명                                                      주거안정장학금
정책키워드명                                                  보조금,교육지원
정책설명내용         원거리 대학 진학으로 인해 주거 관련 비용 부담이 큰 저소득 대학생을 대상으로 주거...
정책대분류명                                                        교육
정책중분류명                                                     교육비지원
정책지원내용         □ 지원금액: 월 최대 20만 원\n※ 단, 방학 중(7~8월, 1~2월)에는 지원...
주관기관코드                                                   1342000
주관기관코드명                  

In [15]:
df_region = pd.read_excel("법정동 기준 시군구 단위.xlsx", sheet_name="통합 버전")
df_region.head()

,시군구,시군구_코드_법정동기준
0,서울 강남구,11680
1,서울 강동구,11740
2,서울 강북구,11305
3,서울 강서구,11500
4,서울 관악구,11620


In [16]:
# 1. 지역 코드-시군구명 매핑 딕셔너리 생성
df_region['시군구_코드_법정동기준'] = df_region['시군구_코드_법정동기준'].astype(str)
region_code_map = df_region.set_index('시군구_코드_법정동기준')['시군구'].to_dict()

# 2. 코드 변환 함수 정의 - 쉼표로 구분된 코드들도 처리
def transform_region_code(code_value):
    # None이나 NaN인 경우 그대로 반환
    if pd.isna(code_value):
        return code_value
        
    # 쉼표로 구분된 여러 코드가 있는지 확인
    if isinstance(code_value, str) and ',' in code_value:
        # 쉼표로 구분된 각 코드를 처리
        codes = code_value.split(',')
        transformed_codes = []
        
        for code in codes:
            code = code.strip()
            # 매핑 딕셔너리에서 시군구명 찾기
            region_name = region_code_map.get(code)
            if region_name:
                transformed_codes.append(region_name)
            else:
                transformed_codes.append(code)
        # 변환된 코드명들을 쉼표로 연결하여 반환
        return ', '.join(transformed_codes) if transformed_codes else code_value
    
    # 단일 코드인 경우    
    else:
        # 매핑 딕셔너리에서 시군구명 찾기
        return region_code_map.get(code_value, code_value)

# 3. 원래 값 저장 (비교용)
original_region_codes = df['정책거주지역코드'].copy()

# 4. 변환 함수 적용하여 시군구명으로 매핑
df['정책거주지역코드'] = df['정책거주지역코드'].apply(transform_region_code)

# 5. 매핑 결과 확인
mapped_count = df['정책거주지역코드'].notna().sum()
print(f"정책거주지역코드: 총 {len(df)}개 중 {mapped_count}개 매핑 완료 ({mapped_count/len(df)*100:.2f}%)")

# 6. 매핑 전후 비교 샘플 출력
sample_df = pd.DataFrame({
    '원래 코드': original_region_codes.head(10),
    '매핑된 시군구명': df['정책거주지역코드'].head(10)
})
print("\n매핑 전후 샘플 비교:")
print(sample_df)

# 7. 매핑된 값들의 분포 확인 (상위 10개)
print("\n지역별 정책 수 (상위 10개):")
print(df['정책거주지역코드'].value_counts().head(10))

정책거주지역코드: 총 3458개 중 3458개 매핑 완료 (100.00%)

매핑 전후 샘플 비교:
                                               원래 코드  \
0  11110,11140,11170,11200,11215,11230,11260,1129...   
1  26110,26140,26170,26200,26230,26260,26290,2632...   
2  11110,11140,11170,11200,11215,11230,11260,1129...   
3  11110,11140,11170,11200,11215,11230,11260,1129...   
4  11110,11140,11170,11200,11215,11230,11260,1129...   
5  26110,26140,26170,26200,26230,26260,26290,2632...   
6  26110,26140,26170,26200,26230,26260,26290,2632...   
7  26110,26140,26170,26200,26230,26260,26290,2632...   
8  26110,26140,26170,26200,26230,26260,26290,2632...   
9  47111,47113,47130,47150,47170,47190,47210,4723...   

                                            매핑된 시군구명  
0  서울 종로구, 서울 중구, 서울 용산구, 서울 성동구, 서울 광진구, 서울 동대문구...  
1  부산 중구, 부산 서구, 부산 동구, 부산 영도구, 부산 부산진구, 부산 동래구, ...  
2  서울 종로구, 서울 중구, 서울 용산구, 서울 성동구, 서울 광진구, 서울 동대문구...  
3  서울 종로구, 서울 중구, 서울 용산구, 서울 성동구, 서울 광진구, 서울 동대문구...  
4  서울 종로구, 서울 중구, 서울 용산구, 서울 성동구, 서울 광진구, 서울 동대문구...

In [17]:
df_region2 = pd.read_csv("법정동코드 전체자료.txt", encoding='cp949', sep='\t')
df_region2.head()

,법정동코드,법정동명,폐지여부
0,1100000000,서울특별시,존재
1,1111000000,서울특별시 종로구,존재
2,1111010100,서울특별시 종로구 청운동,존재
3,1111010200,서울특별시 종로구 신교동,존재
4,1111010300,서울특별시 종로구 궁정동,존재


In [18]:
# 1. 법정동코드 앞 5자리 기준으로 중복을 제거한 데이터프레임 생성
filtered_df_region2 = df_region2[df_region2['폐지여부'] == '존재'].copy()
filtered_df_region2['법정동코드_5자리'] = filtered_df_region2['법정동코드'].astype(str).str[:5]

# 중복된 5자리 코드 중 첫 번째 데이터만 유지
unique_region_df = filtered_df_region2.drop_duplicates(subset=['법정동코드_5자리'])

print(f"원본 데이터 크기: {len(filtered_df_region2)}개")
print(f"중복 제거 후 데이터 크기: {len(unique_region_df)}개")
print(f"제거된 중복 데이터 수: {len(filtered_df_region2) - len(unique_region_df)}개")

# 2. 코드-법정동명 매핑 딕셔너리 생성 
region_code_map = unique_region_df.set_index('법정동코드_5자리')['법정동명'].to_dict()

print(f"법정동 코드 매핑 딕셔너리 크기: {len(region_code_map)}개")
print("법정동 코드 예시:", list(region_code_map.items())[:5])

# 3. 코드 변환 함수는 동일하게 유지
def transform_region_code(code_value):
    # None이나 NaN인 경우 그대로 반환
    if pd.isna(code_value):
        return code_value
        
    # 쉼표로 구분된 여러 코드가 있는지 확인
    if isinstance(code_value, str) and ',' in code_value:
        # 쉼표로 구분된 각 코드를 처리
        codes = code_value.split(',')
        transformed_codes = []
        
        for code in codes:
            code = code.strip()
            # 앞 5자리 추출
            code_5digits = code[:5] if len(code) >= 5 else code
            # 매핑 딕셔너리에서 법정동명 찾기
            region_name = region_code_map.get(code_5digits)
            if region_name:
                transformed_codes.append(region_name)
            else:
                transformed_codes.append(code)
        # 변환된 코드명들을 쉼표로 연결하여 반환
        return ', '.join(transformed_codes) if transformed_codes else code_value
    
    # 단일 코드인 경우    
    else:
        # 앞 5자리 추출
        if isinstance(code_value, str) and len(code_value) >= 5:
            code_5digits = code_value[:5]
            # 매핑 딕셔너리에서 법정동명 찾기
            region_name = region_code_map.get(code_5digits)
            if region_name:
                return region_name
        return code_value

# 4. 원래 값 저장 및 변환 함수 적용
original_region_codes = df['정책거주지역코드'].copy()
df['정책거주지역코드'] = df['정책거주지역코드'].apply(transform_region_code)

# 5. 매핑 결과 확인
mapped_count = (df['정책거주지역코드'] != original_region_codes).sum()
print(f"정책거주지역코드: 총 {len(df)}개 중 {mapped_count}개 매핑 완료 ({mapped_count/len(df)*100:.2f}%)")

# 6. 매핑 전후 비교 샘플 출력
sample_df = pd.DataFrame({
    '원래 코드': original_region_codes.head(10),
    '매핑된 법정동명': df['정책거주지역코드'].head(10)
})
print("\n매핑 전후 샘플 비교:")
print(sample_df)

원본 데이터 크기: 20555개
중복 제거 후 데이터 크기: 280개
제거된 중복 데이터 수: 20275개
법정동 코드 매핑 딕셔너리 크기: 280개
법정동 코드 예시: [('11000', '서울특별시'), ('11110', '서울특별시 종로구'), ('11140', '서울특별시 중구'), ('11170', '서울특별시 용산구'), ('11200', '서울특별시 성동구')]
정책거주지역코드: 총 3458개 중 1808개 매핑 완료 (52.28%)

매핑 전후 샘플 비교:
                                               원래 코드  \
0  서울 종로구, 서울 중구, 서울 용산구, 서울 성동구, 서울 광진구, 서울 동대문구...   
1  부산 중구, 부산 서구, 부산 동구, 부산 영도구, 부산 부산진구, 부산 동래구, ...   
2  서울 종로구, 서울 중구, 서울 용산구, 서울 성동구, 서울 광진구, 서울 동대문구...   
3  서울 종로구, 서울 중구, 서울 용산구, 서울 성동구, 서울 광진구, 서울 동대문구...   
4  서울 종로구, 서울 중구, 서울 용산구, 서울 성동구, 서울 광진구, 서울 동대문구...   
5  부산 중구, 부산 서구, 부산 동구, 부산 영도구, 부산 부산진구, 부산 동래구, ...   
6  부산 중구, 부산 서구, 부산 동구, 부산 영도구, 부산 부산진구, 부산 동래구, ...   
7  부산 중구, 부산 서구, 부산 동구, 부산 영도구, 부산 부산진구, 부산 동래구, ...   
8  부산 중구, 부산 서구, 부산 동구, 부산 영도구, 부산 부산진구, 부산 동래구, ...   
9  경상북도 포항시 남구, 경상북도 포항시 북구, 경상북도 경주시, 경상북도 김천시, ...   

                                            매핑된 법정동명  
0  서울 종로구, 서울 중구, 서울 용산구, 서울 성동구, 서울 광진구, 서울 동대문구...  
1  부산 중

In [35]:
all_region = "서울 종로구, 서울 중구, 서울 용산구, 서울 성동구, 서울 광진구, 서울 동대문구, 서울 중랑구, 서울 성북구, 서울 강북구, 서울 도봉구, 서울 노원구, 서울 은평구, 서울 서대문구, 서울 마포구, 서울 양천구, 서울 강서구, 서울 구로구, 서울 금천구, 서울 영등포구, 서울 동작구, 서울 관악구, 서울 서초구, 서울 강남구, 서울 송파구, 서울 강동구, 부산 중구, 부산 서구, 부산 동구, 부산 영도구, 부산 부산진구, 부산 동래구, 부산 남구, 부산 북구, 부산 해운대구, 부산 사하구, 부산 금정구, 부산 강서구, 부산 연제구, 부산 수영구, 부산 사상구, 부산 기장군, 대구 중구, 대구 동구, 대구 서구, 대구 남구, 대구 북구, 대구 수성구, 대구 달서구, 대구 달성군, 대구광역시 군위군, 인천 중구, 인천 동구, 인천 미추홀구, 인천 연수구, 인천 남동구, 인천 부평구, 인천 계양구, 인천 서구, 인천 강화군, 인천 옹진군, 광주 동구, 광주 서구, 광주 남구, 광주 북구, 광주 광산구, 대전 동구, 대전 중구, 대전 서구, 대전 유성구, 대전 대덕구, 울산 중구, 울산 남구, 울산 동구, 울산 북구, 울산 울주군, 세종특별자치시, 경기도 수원시 장안구, 경기도 수원시 권선구, 경기도 수원시 팔달구, 경기도 수원시 영통구, 경기도 성남시 수정구, 경기도 성남시 중원구, 경기도 성남시 분당구, 경기도 의정부시, 경기도 안양시 만안구, 경기도 안양시 동안구, 경기도 부천시 원미구 , 경기도 부천시 소사구 , 경기도 부천시 오정구 , 경기도 광명시, 경기도 평택시, 경기도 동두천시, 경기도 안산시 상록구, 경기도 안산시 단원구, 경기도 고양시 덕양구, 경기도 고양시 일산동구, 경기도 고양시 일산서구, 경기도 과천시, 경기도 구리시, 경기도 남양주시, 경기도 오산시, 경기도 시흥시, 경기도 군포시, 경기도 의왕시, 경기도 하남시, 경기도 용인시 처인구, 경기도 용인시 기흥구, 경기도 용인시 수지구, 경기도 파주시, 경기도 이천시, 경기도 안성시, 경기도 김포시, 경기도 화성시, 경기도 광주시, 경기도 양주시, 경기도 포천시, 경기도 여주시, 경기도 연천군, 경기도 가평군, 경기도 양평군, 충청북도 청주시 상당구, 충청북도 청주시 서원구, 충청북도 청주시 흥덕구, 충청북도 청주시 청원구, 충청북도 충주시, 충청북도 제천시, 충청북도 보은군, 충청북도 옥천군, 충청북도 영동군, 충청북도 증평군, 충청북도 진천군, 충청북도 괴산군, 충청북도 음성군, 충청북도 단양군, 충청남도 천안시 동남구, 충청남도 천안시 서북구, 충청남도 공주시, 충청남도 보령시, 충청남도 아산시, 충청남도 서산시, 충청남도 논산시, 충청남도 계룡시, 충청남도 당진시, 충청남도 금산군, 충청남도 부여군, 충청남도 서천군, 충청남도 청양군, 충청남도 홍성군, 충청남도 예산군, 충청남도 태안군, 전라남도 목포시, 전라남도 여수시, 전라남도 순천시, 전라남도 나주시, 전라남도 광양시, 전라남도 담양군, 전라남도 곡성군, 전라남도 구례군, 전라남도 고흥군, 전라남도 보성군, 전라남도 화순군, 전라남도 장흥군, 전라남도 강진군, 전라남도 해남군, 전라남도 영암군, 전라남도 무안군, 전라남도 함평군, 전라남도 영광군, 전라남도 장성군, 전라남도 완도군, 전라남도 진도군, 전라남도 신안군, 경상북도 포항시 남구, 경상북도 포항시 북구, 경상북도 경주시, 경상북도 김천시, 경상북도 안동시, 경상북도 구미시, 경상북도 영주시, 경상북도 영천시, 경상북도 상주시, 경상북도 문경시, 경상북도 경산시, 경상북도 의성군, 경상북도 청송군, 경상북도 영양군, 경상북도 영덕군, 경상북도 청도군, 경상북도 고령군, 경상북도 성주군, 경상북도 칠곡군, 경상북도 예천군, 경상북도 봉화군, 경상북도 울진군, 경상북도 울릉군, 경상남도 창원시 의창구, 경상남도 창원시 성산구, 경상남도 창원시 마산합포구, 경상남도 창원시 마산회원구, 경상남도 창원시 진해구, 경상남도 진주시, 경상남도 통영시, 경상남도 사천시, 경상남도 김해시, 경상남도 밀양시, 경상남도 거제시, 경상남도 양산시, 경상남도 의령군, 경상남도 함안군, 경상남도 창녕군, 경상남도 고성군, 경상남도 남해군, 경상남도 하동군, 경상남도 산청군, 경상남도 함양군, 경상남도 거창군, 경상남도 합천군, 제주 제주시, 제주 서귀포시, 강원특별자치도 춘천시, 강원특별자치도 원주시, 강원특별자치도 강릉시, 강원특별자치도 동해시, 강원특별자치도 태백시, 강원특별자치도 속초시, 강원특별자치도 삼척시, 강원특별자치도 홍천군, 강원특별자치도 횡성군, 강원특별자치도 영월군, 강원특별자치도 평창군, 강원특별자치도 정선군, 강원특별자치도 철원군, 강원특별자치도 화천군, 강원특별자치도 양구군, 강원특별자치도 인제군, 강원특별자치도 고성군, 강원특별자치도 양양군, 전북특별자치도 전주시 완산구, 전북특별자치도 전주시 덕진구, 전북특별자치도 군산시, 전북특별자치도 익산시, 전북특별자치도 정읍시, 전북특별자치도 남원시, 전북특별자치도 김제시, 전북특별자치도 완주군, 전북특별자치도 진안군, 전북특별자치도 무주군, 전북특별자치도 장수군, 전북특별자치도 임실군, 전북특별자치도 순창군, 전북특별자치도 고창군, 전북특별자치도 부안군"
all_region

'서울 종로구, 서울 중구, 서울 용산구, 서울 성동구, 서울 광진구, 서울 동대문구, 서울 중랑구, 서울 성북구, 서울 강북구, 서울 도봉구, 서울 노원구, 서울 은평구, 서울 서대문구, 서울 마포구, 서울 양천구, 서울 강서구, 서울 구로구, 서울 금천구, 서울 영등포구, 서울 동작구, 서울 관악구, 서울 서초구, 서울 강남구, 서울 송파구, 서울 강동구, 부산 중구, 부산 서구, 부산 동구, 부산 영도구, 부산 부산진구, 부산 동래구, 부산 남구, 부산 북구, 부산 해운대구, 부산 사하구, 부산 금정구, 부산 강서구, 부산 연제구, 부산 수영구, 부산 사상구, 부산 기장군, 대구 중구, 대구 동구, 대구 서구, 대구 남구, 대구 북구, 대구 수성구, 대구 달서구, 대구 달성군, 대구광역시 군위군, 인천 중구, 인천 동구, 인천 미추홀구, 인천 연수구, 인천 남동구, 인천 부평구, 인천 계양구, 인천 서구, 인천 강화군, 인천 옹진군, 광주 동구, 광주 서구, 광주 남구, 광주 북구, 광주 광산구, 대전 동구, 대전 중구, 대전 서구, 대전 유성구, 대전 대덕구, 울산 중구, 울산 남구, 울산 동구, 울산 북구, 울산 울주군, 세종특별자치시, 경기도 수원시 장안구, 경기도 수원시 권선구, 경기도 수원시 팔달구, 경기도 수원시 영통구, 경기도 성남시 수정구, 경기도 성남시 중원구, 경기도 성남시 분당구, 경기도 의정부시, 경기도 안양시 만안구, 경기도 안양시 동안구, 경기도 부천시 원미구 , 경기도 부천시 소사구 , 경기도 부천시 오정구 , 경기도 광명시, 경기도 평택시, 경기도 동두천시, 경기도 안산시 상록구, 경기도 안산시 단원구, 경기도 고양시 덕양구, 경기도 고양시 일산동구, 경기도 고양시 일산서구, 경기도 과천시, 경기도 구리시, 경기도 남양주시, 경기도 오산시, 경기도 시흥시, 경기도 군포시, 경기도 의왕시, 경기도 하남시, 경기도 용인시 처인구, 경기도 용인시 기흥구, 경기도 용인시 수지구, 경기도 파주시, 경기도 이천시, 경기도 안성시, 경기도

In [38]:
df[df['정책거주지역코드'] == all_region]['정책거주지역코드']

0       서울 종로구, 서울 중구, 서울 용산구, 서울 성동구, 서울 광진구, 서울 동대문구...
19      서울 종로구, 서울 중구, 서울 용산구, 서울 성동구, 서울 광진구, 서울 동대문구...
20      서울 종로구, 서울 중구, 서울 용산구, 서울 성동구, 서울 광진구, 서울 동대문구...
21      서울 종로구, 서울 중구, 서울 용산구, 서울 성동구, 서울 광진구, 서울 동대문구...
22      서울 종로구, 서울 중구, 서울 용산구, 서울 성동구, 서울 광진구, 서울 동대문구...
                              ...                        
3451    서울 종로구, 서울 중구, 서울 용산구, 서울 성동구, 서울 광진구, 서울 동대문구...
3452    서울 종로구, 서울 중구, 서울 용산구, 서울 성동구, 서울 광진구, 서울 동대문구...
3453    서울 종로구, 서울 중구, 서울 용산구, 서울 성동구, 서울 광진구, 서울 동대문구...
3454    서울 종로구, 서울 중구, 서울 용산구, 서울 성동구, 서울 광진구, 서울 동대문구...
3455    서울 종로구, 서울 중구, 서울 용산구, 서울 성동구, 서울 광진구, 서울 동대문구...
Name: 정책거주지역코드, Length: 439, dtype: object

In [39]:
# 정책거주지역코드가 all_region과 일치하는 항목을 '전국'으로 변경
df.loc[df['정책거주지역코드'] == all_region, '정책거주지역코드'] = '전국'

# 변경 결과 확인 
print(f"'전국'으로 변경된 행 수: {(df['정책거주지역코드'] == '전국').sum()}")

# 데이터 몇 건 확인하기
print("\n변경 후 '전국' 값을 가진 데이터 샘플:")
print(df[df['정책거주지역코드'] == '전국'].head(3))

'전국'으로 변경된 행 수: 439

변경 후 '전국' 값을 가진 데이터 샘플:
                    정책번호  기본계획차수  기본계획정책방향번호  기본계획중점과제번호  기본계획과제번호 제공기관그룹코드  \
0   20250521005400110863       1           3          10        23     중앙부처   
19  20250514005400210818       1           2           6        13      지자체   
20  20250514005400110813       1           5          20        52     중앙부처   

   정책제공방법코드 정책승인상태코드         정책명    정책키워드명  ... 등록자최상위기관코드 등록자최상위기관코드명  \
0        기타       승인     주거안정장학금  보조금,교육지원  ...    1492000       고용노동부   
19       기타       승인     대전청년하우스    공공임대주택  ...    1492000       고용노동부   
20     프로그램       승인  유엔참전국 교류캠프      해외진출  ...    1492000       고용노동부   

   정책거주지역코드 정책전공요건코드 정책취업요건코드 정책학력요건코드                 신청기간  \
0        전국     제한없음     제한없음    대학 재학  20250523 ~ 20250623   
19       전국     제한없음      재직자     제한없음                  NaN   
20       전국     제한없음     제한없음    대학 재학  20250507 ~ 20250521   

                 최초등록일시               최종수정일시 정책특화요건코드  
0   2025-05-21 15:36:19  2025-05

In [42]:
df.to_csv('청년정책목록_전체_매핑완료_3.csv', encoding='utf-8', index=False)
df.to_excel('청년정책목록_전체_매핑완료_3.xlsx', index=False)